# ML12 · 进阶选修：表征与生成

从变分自编码器（VAE）出发，建立解耦表征、扩散模型与整个生成模型家族的高层直觉，并学会如何连到实务生成式工作流。本书重观念与直觉，所有资料皆用 numpy 自造，不写完整训练。

## 学习目标
- 理解表征学习（representation learning）的目标：学到「有用、可解释、可迁移」的潜在特征。
- 说明 β-VAE 如何透过调整 KL 权重，在重建与解耦（disentanglement）之间取得平衡。
- 用直觉描述扩散模型（diffusion model）的核心：前向加噪与反向去噪。
- 建立生成模型（generative model）全景图：VAE、GAN、自回归（autoregressive）、扩散模型各自的取舍。
- 能将上述概念连到实务生成式工作流（generative workflow），知道何时用哪一类模型。

In [ ]:
# 概念：环境设置。固定乱数种子让结果可重现，并让 matplotlib 图内嵌显示
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(12)   # 固定种子，使每次运行的随机数据一致，方便对照讲解

# 技巧：图上若要显示中文标题，需指定有中文字体；此处统一用英文标签避免缺字方框
plt.rcParams["axes.unicode_minus"] = False   # 避免负号显示成方框

print("numpy version:", np.__version__)
print("环境就绪，后续数据皆以此 rng 自造")

## 表征学习的概念与用途

表征学习（representation learning）是指让模型自己学出一组「描述数据的座标系」，而不是直接背单一任务的答案。学到的这组座标就是潜在空间（latent space）。

为什么要学：一组好的表征能让下游任务（downstream task）如分类、聚类、检索都变简单；同一组特征还能迁移（transfer）到不同任务。特征抽取（feature extraction）就是把高维原始数据压成这种低维、好用的座标。

判断表征好坏的直觉：相似的数据在潜在空间里会自然靠在一起。

In [ ]:
# 概念：把高维「建筑量体」特征压到 2 维潜在座标，观察相似量体是否自然聚在一起
# 造一批假建筑量体：每栋有 楼高(公尺) / 面积(平方公尺) / 开窗比例(0~1) 三个特征
# 刻意分成两种典型：高瘦高层、矮胖低层，看它们在潜在空间是否分开
tall_thin = rng.normal(loc=[80, 90, 0.6], scale=[8, 10, 0.05], size=(15, 3))
short_fat = rng.normal(loc=[15, 220, 0.3], scale=[3, 25, 0.05], size=(15, 3))
volumes = np.vstack([tall_thin, short_fat])   # 叠成 30 栋 × 3 特征的矩阵

# 先标准化（逐栏 z-score），让不同数量级的特征公平参与压缩
z = (volumes - volumes.mean(axis=0)) / volumes.std(axis=0)

# 概念：用 PCA 直觉做最简降维——取共变异数矩阵特征矢量的前两个主轴
cov = np.cov(z, rowvar=False)              # rowvar=False：每栏是一个变量（特征）
eigvals, eigvecs = np.linalg.eigh(cov)     # eigh 适用对称矩阵，回传由小到大的特征值
top2 = eigvecs[:, ::-1][:, :2]             # 反转取最大两个方向，即信息量最大的两个主轴
latent = z @ top2                          # 投影到 2 维潜在座标

print("原始特征 shape:", volumes.shape, " -> 潜在座标 shape:", latent.shape)

plt.figure(figsize=(5, 4))
plt.scatter(latent[:15, 0], latent[:15, 1], label="tall-thin")
plt.scatter(latent[15:, 0], latent[15:, 1], label="short-fat")
plt.xlabel("latent dim 1"); plt.ylabel("latent dim 2")
plt.title("Building volumes in 2D latent space"); plt.legend()
plt.tight_layout(); plt.show()

## 从 VAE 到 β-VAE：用 KL 权重换取解耦

变分自编码器（VAE, variational autoencoder）把数据压进一个几率化的潜在空间，再从中重建回来。它的损失有两部分：重建损失（reconstruction loss）要求还原得像，KL 散度（KL divergence）要求潜在分布贴近标准常态。

β-VAE 在 KL 项前加一个权重 β：loss = 重建 + β · KL。β 变大会压缩潜在信息、逼每个维度更独立（更解耦），但重建会变糊。这是一场「解耦（disentanglement）vs 还原」的拉锯。

公式（仅示意，不训练）：loss = reconstruction + β · KL。

In [ ]:
# 概念：用自造的两个生成因子（楼高、胖瘦）数据，示意 β 由小到大时的「解耦 vs 重建」取舍
# 造两个彼此独立的真实生成因子：height（楼高）与 chubby（胖瘦）
n = 200
height = rng.uniform(-1, 1, size=n)        # 真实因子 1
chubby = rng.uniform(-1, 1, size=n)        # 真实因子 2（与因子 1 独立）
factors = np.stack([height, chubby], axis=1)

# 用一个简化模型示意：β 越大，潜在维度越「对齐单一因子」但重建误差越高
# alignment 越接近 1 代表某潜在维度几乎只反映单一生成因子（更解耦）
def simulate_beta(beta):
    align = 0.5 + 0.45 * (1 - np.exp(-beta))        # 随 β 增大趋近于高度对齐
    recon_err = 0.05 + 0.25 * (1 - np.exp(-beta))   # 注意：β 越大重建越糊（误差升高）
    return align, recon_err

print("beta  | 因子对齐度 | 重建误差")
for beta in [0.5, 1.0, 4.0, 8.0]:
    align, err = simulate_beta(beta)
    print(f"{beta:>4.1f}  |   {align:.3f}   |  {err:.3f}")

betas = np.linspace(0.1, 10, 50)
aligns = [simulate_beta(b)[0] for b in betas]
errs = [simulate_beta(b)[1] for b in betas]
plt.figure(figsize=(5, 4))
plt.plot(betas, aligns, label="disentanglement (alignment)")
plt.plot(betas, errs, label="reconstruction error")
plt.xlabel("beta (KL weight)"); plt.ylabel("score")
plt.title("beta-VAE trade-off (illustrative)"); plt.legend()
plt.tight_layout(); plt.show()

## 解读与评估解耦表征

解耦的价值在于「转一个旋钮只改一件事」：理想上每个潜在维度只控制一个语意因子。要检查这件事，最直观的方法是潜在遍历（latent traversal）。

潜在遍历：固定其他维度，只扫动单一潜在维度，看输出怎么变。若扫动某维度时只有「楼高」单调改变、其他属性不动，代表这个维度与该因子对齐（factor alignment）良好，可解释性（interpretability）高。

何时用：在比较不同模型或不同 β 设置时，遍历图是判断解耦好坏最直接的工具。

In [ ]:
# 概念：潜在遍历——固定其他维度、只扫动单一维度，观察是否只有目标属性改变
# 用一个示意解码器把 2 维潜在座标 (z0, z1) 映回三个可观察属性
# 设计成「解耦良好」：z0 几乎只决定楼高，z1 几乎只决定胖瘦
def decode(z0, z1):
    height = 10 + 35 * (z0 + 1)        # 楼高主要由 z0 控制
    width = 8 + 4 * (z1 + 1)           # 宽幅（胖瘦）主要由 z1 控制
    window = 0.4 + 0.02 * z1           # 开窗比例只受 z1 微弱影响
    return height, width, window

sweep = np.linspace(-1, 1, 9)          # 扫动范围：把单一维度从 -1 拉到 1
fixed_z1 = 0.0                         # 注意：遍历时其他维度必须固定，才能归因到被扫的维度
heights = [decode(z0, fixed_z1)[0] for z0 in sweep]
widths = [decode(z0, fixed_z1)[1] for z0 in sweep]

print("扫动 z0（固定 z1）时的楼高:", np.round(heights, 1))
print("扫动 z0（固定 z1）时的宽幅:", np.round(widths, 1), "（应几乎不变）")

plt.figure(figsize=(5, 4))
plt.plot(sweep, heights, marker="o", label="height (should change)")
plt.plot(sweep, widths, marker="s", label="width (should stay flat)")
plt.xlabel("z0 (traversed latent dim)"); plt.ylabel("attribute value")
plt.title("Latent traversal: one knob, one change"); plt.legend()
plt.tight_layout(); plt.show()

## 扩散模型的高层直觉

扩散模型（diffusion model）把生成想成两个方向相反的过程。前向过程（forward process）一步步把干净数据加上高斯杂讯，最后变成几乎纯杂讯；反向过程（reverse process）则学会一步步去噪，把杂讯还原回数据。

关键直觉：生成是渐进的去噪步骤（denoising steps），不是一次到位。与 VAE 对照——VAE 用一次编码/解码往返，扩散则用很多小步累积。

本节用一维信号示意加噪与去噪的方向性，不写真正的训练。

In [ ]:
# 概念：对一条一维「沿街天际线高度曲线」逐步加噪（前向），再示意逐步去噪（反向）
x = np.linspace(0, 4 * np.pi, 120)
skyline = 30 + 12 * np.sin(x) + 5 * np.sin(2.7 * x)   # 自造一条起伏的天际线高度

# 前向过程：逐步注入高斯杂讯，越后面的步骤越接近纯杂讯
steps = 4
noisy_versions = [skyline]
current = skyline.copy()
for t in range(steps):
    current = current + rng.normal(0, 4, size=skyline.shape)   # 每步叠加一点杂讯
    noisy_versions.append(current.copy())

# 反向过程（示意）：用简单移动平均当「去噪器」，一步步把杂讯版本拉回平滑
def denoise_once(signal, k=7):
    kernel = np.ones(k) / k                       # 注意：这只是示意去噪，真正模型是学出来的
    return np.convolve(signal, kernel, mode="same")

recovered = noisy_versions[-1].copy()
for _ in range(steps):
    recovered = denoise_once(recovered)           # 反复去噪，逐步逼近原信号

print("原始与最终去噪结果的均方差:", round(float(np.mean((skyline - recovered) ** 2)), 2))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(skyline, label="clean")
axes[0].plot(noisy_versions[-1], alpha=0.7, label="after forward noising")
axes[0].set_title("Forward: clean -> noise"); axes[0].legend()
axes[1].plot(skyline, label="clean")
axes[1].plot(recovered, alpha=0.8, label="after reverse denoising")
axes[1].set_title("Reverse: noise -> recovered"); axes[1].legend()
plt.tight_layout(); plt.show()

## 生成模型全景与取舍

生成模型（generative model）是一个家族，常见四类：VAE、GAN、自回归（autoregressive）、扩散模型。没有万用模型，差别在取舍。

用四个面向比较选型：样本品质（quality）、多样性（diversity）、可控性（controllability）、训练稳定度（stability）。例如 GAN 品质高但训练不稳；扩散品质与多样性都好但生成慢；VAE 稳定可控但样本偏糊；自回归品质好但逐元素生成慢。

下一个 cell 用一张自造的对照表整理相对强弱与各自适用情境。

In [ ]:
# 概念：用字典/list 自造一张四类生成模型的四面向比较表，并各举一个适用情境
# 分数为相对示意（1=弱, 5=强），非实测数据，用来创建选型直觉
models = ["VAE", "GAN", "Autoregressive", "Diffusion"]
scores = {
    "quality":         [3, 5, 4, 5],
    "diversity":       [4, 2, 4, 5],
    "controllability": [4, 3, 3, 4],
    "stability":       [5, 2, 4, 4],
}
use_case = {
    "VAE": "需要平滑可控潜在空间、能做表征与插值",
    "GAN": "追求最锐利逼真的单张样本、可接受训练调参成本",
    "Autoregressive": "串行型数据、逐步生成且要高保真",
    "Diffusion": "要高品质又高多样性、可接受较慢生成",
}

header = "model           | qual | dive | ctrl | stab"
print(header); print("-" * len(header))
for i, m in enumerate(models):
    # 技巧：用 f-string 对齐字段，纯文字表也能读得清楚
    print(f"{m:<15} |  {scores['quality'][i]}   |  {scores['diversity'][i]}   |"
          f"  {scores['controllability'][i]}   |  {scores['stability'][i]}")

print("\n各自适用情境:")
for m in models:
    print(f"  - {m}: {use_case[m]}")

## 实务生成式工作流

实务上很少只用单一模型，而是一条生成式工作流（generative workflow）：表征 + 生成 + 条件控制 + 评估。理解概念才能在工具链中做对选择与把关。

几个关键环节：条件生成（conditional generation）让你用条件向量指定想要的属性；潜在空间操控（latent manipulation）让你在潜在座标上微调语意；最后一定要评估与人工把关（human-in-the-loop），由人对候选结果排序与筛选。

下一个 cell 用一条最小工作流骨架（pseudo-pipeline）示意：从条件向量到生成候选、再到评分排序与人工把关，资料与分数皆自造。

In [ ]:
# 概念：最小生成式工作流骨架——条件矢量 -> 生成候选 -> 评分排序 -> 人工把关
# 条件矢量：指定想要的目标属性（此例：偏好高楼、适中开窗比例）
condition = {"target_height": 60.0, "target_window": 0.5}

# 步骤 1：用条件加随机扰动生成一批候选量体（示意「条件生成」）
def generate_candidates(cond, k=5):
    base = np.array([cond["target_height"], cond["target_window"]])
    noise = rng.normal(0, [10, 0.1], size=(k, 2))   # 在条件附近采样多个候选
    return base + noise

# 步骤 2：对每个候选评分（示意评估）——越贴近条件分数越高
def score(candidate, cond):
    target = np.array([cond["target_height"], cond["target_window"]])
    dist = np.linalg.norm((candidate - target) / np.array([20, 0.2]))   # 范式距离
    return float(np.exp(-dist))                                          # 距离越小分数越接近 1

candidates = generate_candidates(condition)
scores = np.array([score(c, condition) for c in candidates])

# 步骤 3：依分数由高到低排序（自动排序，供人工把关）
order = np.argsort(scores)[::-1]
print("候选量体 (height, window) 与分数，已依分数排序:")
for rank, idx in enumerate(order, start=1):
    h, w = candidates[idx]
    print(f"  rank {rank}: height={h:5.1f}, window={w:.2f}, score={scores[idx]:.3f}")

# 步骤 4：人工把关——人通常只从前几名挑选，这里示意取 top-2 交付审查
print("\n交付人工审查的 top-2 候选 index:", order[:2].tolist())

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）量体的潜在地图（集成：表征学习 + 潜在空间）
#   情境：你有一批自造的都市建筑量体（楼高、基地面积、容积率），想看看它们在潜在空间中怎么分布。
#   要求：
#     1. 用 numpy 自造约 30 笔建筑量体矢量（可刻意分成「高瘦高层」与「矮胖低层」两种典型）。
#     2. 先逐栏标准化，再把它们投影到 2 维潜在座标（简单降维或自订压缩皆可），画出散布图。
#   验收：应该看到相似量体在潜在地图上分成可辨识的群（两个明显群落）。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）解耦旋钮测试（集成：beta-VAE + KL 权重 + 解耦 + 潜在遍历）
#   情境：你想验证调大 KL 权重 beta 是否真的让「楼高」变成一个独立可控的旋钮。
#   要求：
#     1. 自造带有两个生成因子（楼高、胖瘦）的量体数据。
#     2. 用两组不同 beta（小、大）的示意设置，比较重建误差与潜在维度对齐程度（可用数值表）。
#     3. 对较佳的一组做潜在遍历：固定其他维度、只扫动一个维度，观察是否只有楼高改变。
#   验收：应看到大 beta 时重建较糊但某维度几乎只控制楼高，呈现解耦与重建的取舍。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）为都市量体选一条生成路线（集成：扩散模型直觉 + 生成模型全景 + 生成式工作流 + 表征）
#   情境：团队要替「沿街天际线量体」做生成式发想，需在 VAE、GAN、自回归、扩散模型中选型并设计最小工作流。
#   要求：
#     1. 自造一条一维天际线高度曲线，示意对它前向加噪再反向去噪的方向（画图或印均方差皆可）。
#     2. 用四面向（品质、多样性、可控性、稳定度）比较表为此任务挑选一类生成模型，并写出选型理由。
#     3. 设计一个「条件矢量 -> 生成候选 -> 评分排序 -> 人工把关」的最小工作流骨架（流程与假分数自造）。
#   验收：应看到一条有理由的选型结论，加上一个可解释、含评估把关的生成式工作流草图。
# TODO: 在这里完成


## 小结

- 表征学习的目标是学出「数据的内在座标系」（潜在空间），让分类、聚类、检索等下游任务更简单，且能迁移；好表征的直觉是相似数据自然靠在一起。
- VAE 的损失是重建 + KL；β-VAE 在 KL 前加权重 β，β 变大更解耦但重建变糊，是「解耦 vs 还原」的拉锯。
- 评估解耦最直接的工具是潜在遍历：固定其他维度、只扫动一个维度，看是否「一个旋钮只改一件事」。
- 扩散模型把生成拆成前向加噪与反向去噪的渐进步骤，与 VAE 的一次往返形成对照。
- 生成模型没有万用解；用品质、多样性、可控性、稳定度四面向比较 VAE / GAN / 自回归 / 扩散，创建选型直觉。
- 实务是一条工作流：表征 + 生成 + 条件控制 + 评估；条件生成与潜在操控决定输出，人工把关负责最终筛选与排序。